# Phase 6b · step 5 — the unblinding

Step 5 of `preregistration/ANALYSIS_PLAN.md` §10: **the labels are joined, once**, and
every outcome in §§4–8 is computed for both analyses:

| | Status |
|---|---|
| **Registered** analysis | **the primary result** — decides H1, H1′, H2, H3 |
| **Amended** analysis (D12, D13) | reported beside it, as a deviation analysis |

H1 is decided on the EyePACS test split (custom, D8), H1′ — the thesis — on APTOS **and**
Messidor-2, both. Every claim needs all three seeds positive, every seed's paired 95%
interval clear of zero, and the mean effect larger than the seed spread (§8).

## Run it once

Everything here is deterministic: the bootstrap is seeded, and the parameters, code and
predictions are fixed by commit and digest. If the session is interrupted, re-run from the
top and the same numbers come out — a re-run repeats the computation; it is not a second
look. Nothing may change between the first run and any repeat.

## Inputs

`verify-dr-locked` (09's output) · `verify-dr-fitted` (08's output: the OOD statistics) ·
`verify-dr-manifests`.

| Setting | Value |
|---|---|
| Accelerator | **None** — CPU |
| Persistence | Files only |
| Internet | On |

About 30–60 minutes: two analyses per dataset, 2,000 bootstrap resamples each, with EM
re-run inside every resample on the externals.

## 1 · Clone the repo

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
sys.path.insert(0, str(REPO_DIR / "src"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Drop verify_dr modules left over from an earlier checkout in this kernel.
# Python caches modules by NAME, not by file, so re-cloning mid-session does
# nothing for an already-imported package: a later cell importing a function
# added upstream still fails with ImportError against the new files on disk.
for _stale in [m for m in list(sys.modules)
               if m == "verify_dr" or m.startswith("verify_dr.")]:
    del sys.modules[_stale]

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

## 2 · Step 3 happened: the fitted parameters are committed

In [ ]:
# Plan s10, step 3 must come first: the fitted parameters committed and their digest
# recorded in PREREGISTRATION.md. This refuses before a single locked image is read.
from verify_dr.triage.params import step3_record

PARAMS = step3_record(REPO_DIR)
print("step 3 verified: preregistration/fitted_params.json is committed, unmodified,")
print("and its digest is recorded in PREREGISTRATION.md")
print("  digest  ", PARAMS["digest"])
print("  analyses", "registered (primary) + amended (D12, D13)" if "amended" in PARAMS
      else "registered only")

## 3 · Find the locked pass, the fitted statistics and the manifests

In [ ]:
import json, shlex, subprocess
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    """Run a script, streaming its output as it arrives; raise if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"exit {code}")

def find_locked():
    """The locked pass: a directory holding messidor2/, aptos/ and eyepacs_test/."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for hit in sorted(base.rglob("eyepacs_test/run.json")):
            root = hit.parent.parent
            if all((root / job / "run.json").exists() for job in ("aptos", "messidor2")):
                return root
    return None

def find_fitted():
    """08's output: the directory whose ood/ holds every model's statistics."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for hit in sorted(base.rglob("ood")):
            if all((hit / PARAMS["models"][m]["ood"]["file"].split("/", 1)[1]).exists()
                   for m in PARAMS["models"]):
                return hit.parent
    return None

def find_manifest_dir():
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        hits = sorted(base.rglob("dataset_plan.json"))
        if hits:
            return hits[0].parent
    return None

LOCKED, FITTED, MANIFEST_DIR = find_locked(), find_fitted(), find_manifest_dir()
if LOCKED is None:
    raise RuntimeError("No complete locked pass found. Attach verify-dr-locked (09's output).")
if FITTED is None:
    raise RuntimeError("No OOD statistics found. Attach verify-dr-fitted (08's output).")
if MANIFEST_DIR is None:
    raise RuntimeError("No manifests found. Attach verify-dr-manifests.")

EXPECTED = {"messidor2": 1744, "aptos": 3662, "eyepacs_test": 17615}
for job, rows in EXPECTED.items():
    images = LOCKED / job / "images.csv"
    if not images.exists():
        raise RuntimeError(f"{job} is incomplete in {LOCKED}: finish 09 first.")
    with open(images) as fh:
        n = sum(1 for _ in fh) - 1
    print(f"  {job:<14} {n:>6} images" + ("" if n == rows else f"   <-- expected {rows}"))
print("locked pass :", LOCKED)
print("fitted stats:", FITTED)
print("manifests   :", MANIFEST_DIR)

RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

## 4 · The unblinding

`analyse.py --unblind` joins the labels for one dataset and computes every table for both
analyses. It refuses unless the parameters are committed, and unless the pass came from
the fitted checkpoints with the fitted constants.

In [ ]:
PARAMS_PATH = REPO_DIR / "preregistration" / "fitted_params.json"
JOBS = [
    # name, manifest, role
    ("messidor2", MANIFEST_DIR / "messidor2_external.csv", "external"),
    ("aptos", MANIFEST_DIR / "aptos_external.csv", "external"),
    ("eyepacs_test", MANIFEST_DIR / "eyepacs_full.csv", "in_domain"),
]
for name, manifest, role in JOBS:
    print("\n" + "=" * 72 + f"\n{name}\n" + "=" * 72, flush=True)
    run(" ".join([f"python {q(REPO_DIR / 'scripts/analyse.py')} dataset",
                  f"--pass-dir {q(LOCKED / name)}", f"--labels {q(manifest)}", "--split test",
                  f"--fitted {q(PARAMS_PATH)}", f"--ood-dir {q(FITTED)}",
                  f"--name {name} --role {role} --unblind",
                  f"--out {q(RESULTS / name)}"]))

## 5 · The verdicts

In [ ]:
run(" ".join([f"python {q(REPO_DIR / 'scripts/analyse.py')} verdicts",
              f"--in-domain {q(RESULTS / 'eyepacs_test' / 'results.json')}",
              f"--external {q(RESULTS / 'aptos' / 'results.json')} "
              f"{q(RESULTS / 'messidor2' / 'results.json')}",
              f"--out {q(RESULTS)}"]))

## 6 · The readout

The numbers the write-up needs, per dataset and analysis: every arm's coverage–accuracy
AUC, the per-seed effects with their intervals, calibration, and M3 against the truth.

In [ ]:
ARMS = ("none", "confidence", "ood", "disagreement", "combined")

def effects_line(entry):
    seeds = "  ".join(f"s{s} {d['effect']:+.4f} [{d['interval'][0]:+.4f}, {d['interval'][1]:+.4f}]"
                      for s, d in entry["per_seed"].items())
    return f"{seeds}  -> {'SUPPORTED' if entry['supported'] else 'not supported'}"

for name, *_ in JOBS:
    R = json.loads((RESULTS / name / "results.json").read_text())
    print("#" * 78 + f"\n{name}: {R['n']} images, role {R['role']}, true grades "
          f"{R['true_grade_distribution']}\n" + "#" * 78)
    for label, block in (("REGISTERED (primary)", R), ("AMENDED (D12, D13)", R.get("amended"))):
        if block is None:
            continue
        ev = block["evidence"]
        print(f"\n--- {label} ---")
        pct = lambda v: "n/a" if v is None else f"{v:.1%}"
        print(f"M3: QWK {ev['m3_qwk']:.3f}; evidence in {pct(ev['evidence_any_given_grade0'])} of "
              f"grade-0 images; none in {pct(ev['evidence0_given_referable'])} of referable; "
              f"confusion {ev['confusion_true_by_evidence']}")
        for model in sorted(block["models"], key=lambda n: ("_ddr_" in n, n)):
            r = block["models"][model]
            a = r["arms"]
            c = r["calibration"]
            base = c["h2_base"]
            line = (f"{model:<26} acc {r['accuracy']:.4f} QWK {r['qwk']:.4f} | AUC "
                    + " ".join(f"{k} {a[k]['auc']:.4f}" for k in ARMS)
                    + f" | ECE raw {c['raw']['ece']:.4f} {base} {c[base]['ece']:.4f}")
            if "em" in c:
                line += f" EM {c['em']['ece']:.4f} oracle {c['oracle']['ece']:.4f}"
            print(line)
        for variant, entry in block["claims_on_this_dataset"].items():
            if variant == "H3_qwk_gain":
                print(f"  H3 qwk gain           {effects_line(entry)}")
                continue
            for key, sub in entry.items():
                print(f"  {variant:<17} {key:<9} {effects_line(sub)}")
    print()

---
## 7 · Save, and what to paste back

1. **Save Version → Quick Save.**
2. Publish `/kaggle/working/results` as **`verify-dr-results`**: `results.json`,
   `curves.npz` per dataset, and `verdicts.json`. They are the record the write-up cites.
3. Paste back the outputs of **sections 5 and 6**.